In [24]:
%pip install datasets transformers torch scikit-learn pandas accelerate


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from sklearn.metrics import classification_report


In [26]:
dataset = load_dataset("go_emotions")

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})


In [27]:
label_names = dataset["train"].features["labels"].feature.names
print(label_names)


['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral']


In [28]:
target_emotions = ["sadness", "fear", "anger", "joy", "neutral"]

target_indices = [label_names.index(emotion) for emotion in target_emotions]

print("Selected Emotion Indices:", target_indices)


Selected Emotion Indices: [25, 14, 2, 17, 27]


In [29]:
def convert_labels(example):
    for label in example["labels"]:
        if label in target_indices:
            example["label"] = target_indices.index(label)
            return example
    return None

filtered_train = dataset["train"].map(convert_labels)
filtered_train = filtered_train.filter(lambda x: x["label"] is not None)

print("Filtered Train Size:", len(filtered_train))


Map:   0%|          | 0/43410 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18906 [00:00<?, ? examples/s]

Filtered Train Size: 18906


In [30]:
filtered_test = dataset["test"].map(convert_labels)
filtered_test = filtered_test.filter(lambda x: x["label"] is not None)

print("Filtered Test Size:", len(filtered_test))


Map:   0%|          | 0/5427 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2359 [00:00<?, ? examples/s]

Filtered Test Size: 2359


In [31]:
filtered_train = filtered_train.select(range(8000))
filtered_test = filtered_test.select(range(2000))

print("Reduced Train Size:", len(filtered_train))
print("Reduced Test Size:", len(filtered_test))


Reduced Train Size: 8000
Reduced Test Size: 2000


In [32]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [33]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding="max_length")

train_dataset = filtered_train.map(tokenize_function, batched=True)
test_dataset = filtered_test.map(tokenize_function, batched=True)


Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [34]:
train_dataset = train_dataset.remove_columns(["text", "labels"])
test_dataset = test_dataset.remove_columns(["text", "labels"])

train_dataset.set_format("torch")
test_dataset.set_format("torch")


In [35]:
training_args = TrainingArguments(
    output_dir="../results",
    num_train_epochs=1,  # 1 epoch for speed
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_steps=100
)


In [36]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)


In [37]:
trainer.train()


c:\Users\avina\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
100,0.826145
200,0.514813
300,0.493414
400,0.401200
500,0.407289


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.5285721817016602, metrics={'train_runtime': 10715.4168, 'train_samples_per_second': 0.747, 'train_steps_per_second': 0.047, 'total_flos': 1059795886080000.0, 'train_loss': 0.5285721817016602, 'epoch': 1.0})

In [38]:
predictions = trainer.predict(test_dataset)

y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids

print(classification_report(y_true, y_pred))


c:\Users\avina\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

           0       0.74      0.61      0.67       121
           1       0.75      0.65      0.69        65
           2       0.67      0.53      0.59       173
           3       0.84      0.80      0.82       144
           4       0.90      0.94      0.92      1497

    accuracy                           0.87      2000
   macro avg       0.78      0.71      0.74      2000
weighted avg       0.86      0.87      0.86      2000



In [39]:
model.save_pretrained("../saved_models/emotion_model")
tokenizer.save_pretrained("../saved_models/emotion_model")

print("Model saved successfully!")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [40]:
emotion_labels = ["sadness", "fear", "anger", "joy", "neutral"]

def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits, dim=1).item()
    return emotion_labels[prediction]

print(predict_emotion("I feel stressed about exams"))
print(predict_emotion("I am very happy today"))
print(predict_emotion("I am scared about tomorrow"))


sadness
joy
fear


In [42]:
import torch.nn.functional as F

def predict_emotion_with_confidence(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = F.softmax(outputs.logits, dim=1)
    prediction = torch.argmax(probs, dim=1).item()
    confidence = probs[0][prediction].item()
    
    return emotion_labels[prediction], confidence

while True:
    user_input = input("You: ")
    
    if user_input.lower() == "exit":
        break
        
    emotion, confidence = predict_emotion_with_confidence(user_input)
    print(f"Emotion: {emotion} | Confidence: {confidence:.2f}")


Emotion: joy | Confidence: 0.94
